# Single-Class YOLO Training - Micro Dataset (Kaggle)

This notebook trains YOLO for one class: `microplastic`, using your augmented dataset
from `data/micro/yolo_single_aug` uploaded to Kaggle.

It follows the same logic as your reference notebook:
- auto-detect dataset under `/kaggle/input`
- copy to `/kaggle/working`
- fix `dataset.yaml` path
- train + validate + export

## 1. Install Dependencies

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.1.0', 'pyyaml'], check=True)

import ultralytics
print('Ultralytics:', ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.21


## 2. GPU Check + Reproducibility

In [2]:
import os
import random
import numpy as np
import torch
import psutil

assert torch.cuda.is_available(), 'No GPU detected. In Kaggle set Accelerator to GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('RAM GB:', f"{psutil.virtual_memory().total / 1e9:.1f}")
print('Disk free GB:', f"{psutil.disk_usage('/kaggle/working').free / 1e9:.1f}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
print('Seed:', SEED)

GPU: Tesla T4
PyTorch: 2.9.0+cu126
CUDA: 12.6
RAM GB: 33.7
Disk free GB: 20.9
Seed: 42


## 3. Auto-Discover Dataset and Copy to Working Directory

Expected dataset structure (uploaded to Kaggle):
- dataset.yaml
- images/train, images/val
- labels/train, labels/val

Tip: name your Kaggle dataset with `micro` and/or `single` in its path for best auto-detection.

In [3]:
import os
import shutil
import yaml
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
LOCAL_DATASET_PATH = Path('/kaggle/working/dataset_micro_single_aug')
OUTPUT_PATH = Path('/kaggle/working/experiments/yolo')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print('Input folders:', sorted([p.name for p in INPUT_ROOT.iterdir()]))

candidates = []
for yml in INPUT_ROOT.rglob('dataset.yaml'):
    parent = yml.parent
    if (parent / 'images').exists() and (parent / 'labels').exists():
        score = 0
        pstr = str(parent).lower()
        if 'micro' in pstr:
            score += 2
        if 'single' in pstr:
            score += 2
        if 'aug' in pstr:
            score += 1
        candidates.append((score, parent))

assert candidates, 'No valid YOLO dataset found. Need dataset.yaml + images/ + labels/ under /kaggle/input.'
candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
src = candidates[0][1]
print('Using source:', src)

if LOCAL_DATASET_PATH.exists():
    # self-heal stale copies
    if not (LOCAL_DATASET_PATH / 'dataset.yaml').exists() or not (LOCAL_DATASET_PATH / 'images' / 'train').exists():
        shutil.rmtree(LOCAL_DATASET_PATH)

if not LOCAL_DATASET_PATH.exists():
    shutil.copytree(str(src), str(LOCAL_DATASET_PATH))
    print('Copied to:', LOCAL_DATASET_PATH)
else:
    print('Already exists and valid:', LOCAL_DATASET_PATH)

YAML_PATH = LOCAL_DATASET_PATH / 'dataset.yaml'
assert YAML_PATH.exists(), f'dataset.yaml missing at {YAML_PATH}'

with open(YAML_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(LOCAL_DATASET_PATH)
with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('Updated dataset.yaml path ->', cfg['path'])
print('Class config:', cfg.get('names'))

Input folders: ['datasets']
Using source: /kaggle/input/datasets/mpdetect/yolo-single-augmented-micro
Copied to: /kaggle/working/dataset_micro_single_aug
Updated dataset.yaml path -> /kaggle/working/dataset_micro_single_aug
Class config: {0: 'microplastic'}


## 4. Train YOLO (Single-Class microplastic)

In [4]:
from ultralytics import YOLO
from pathlib import Path
import datetime
import shutil

MODEL = 'yolov8m.pt'
IMGSZ = 1280
BATCH_SIZE = 2
EPOCHS = 200
PATIENCE = 50
EXPERIMENT = 'mp_yolov8m_single_class_micro'
BACKUP_EVERY = 50
BACKUP_ROOT = Path('/kaggle/working/backups/yolo_micro')

def auto_backup(trainer):
    epoch = trainer.epoch + 1
    if epoch % BACKUP_EVERY != 0:
        return
    weights_dir = Path(trainer.save_dir) / 'weights'
    backup_dir = BACKUP_ROOT / f'epoch_{epoch:04d}'
    backup_dir.mkdir(parents=True, exist_ok=True)
    for name in ('best.pt', 'last.pt'):
        src = weights_dir / name
        if src.exists():
            shutil.copy2(src, backup_dir / name)
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[AUTO-BACKUP] epoch {epoch} -> {backup_dir} ({ts})')

model = YOLO(MODEL)
model.add_callback('on_train_epoch_end', auto_backup)

print('=' * 70)
print('Training YOLO micro single-class')
print('data:', str(YAML_PATH))
print('project:', str(OUTPUT_PATH))
print('name:', EXPERIMENT)
print('=' * 70)

results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    workers=2,
    seed=SEED,
    deterministic=True,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=5.0,
    cos_lr=True,
    patience=PATIENCE,
    augment=True,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    close_mosaic=20,
    fliplr=0.5,
    flipud=0.5,
    degrees=15.0,
    translate=0.2,
    scale=0.5,
    shear=5.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    amp=True,
    cache='disk',
    multi_scale=False,
    project=str(OUTPUT_PATH),
    name=EXPERIMENT,
    exist_ok=True,
    save=True,
    save_period=25,
    plots=True
)

print('Best:', OUTPUT_PATH / EXPERIMENT / 'weights' / 'best.pt')
print('Last:', OUTPUT_PATH / EXPERIMENT / 'weights' / 'last.pt')

Training YOLO micro single-class
data: /kaggle/working/dataset_micro_single_aug/dataset.yaml
project: /kaggle/working/experiments/yolo
name: mp_yolov8m_single_class_micro
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_micro_single_aug/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mod

## 5. Validate Best Model

In [5]:
from ultralytics import YOLO
from pathlib import Path

BEST = OUTPUT_PATH / EXPERIMENT / 'weights' / 'best.pt'
if not BEST.exists():
    backups = sorted(BACKUP_ROOT.glob('epoch_*/best.pt')) if BACKUP_ROOT.exists() else []
    assert backups, 'best.pt not found in main or backup paths.'
    BEST = backups[-1]

print('Validating:', BEST)
m = YOLO(str(BEST))
metrics = m.val(data=str(YAML_PATH), imgsz=IMGSZ, batch=BATCH_SIZE, conf=0.001, iou=0.6, plots=True)

print('mAP50:', f"{metrics.box.map50:.4f}")
print('mAP50-95:', f"{metrics.box.map:.4f}")
print('Precision:', f"{metrics.box.mp:.4f}")
print('Recall:', f"{metrics.box.mr:.4f}")

Validating: /kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights/best.pt
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 156.5±69.0 MB/s, size: 1287.8 KB)
val: Scanning /kaggle/working/dataset_micro_single_aug/labels/val.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 83.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 5.6it/s 18.0s
                   all        200        622      0.829       0.82      0.824      0.396
Speed: 3.1ms preprocess, 81.4ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP50: 0.8245
mAP50-95: 0.3961
Precision: 0.8286
Recall: 0.8199


## 6. Export and Download

In [6]:
from ultralytics import YOLO
from pathlib import Path
import shutil

BEST = OUTPUT_PATH / EXPERIMENT / 'weights' / 'best.pt'
if not BEST.exists():
    backups = sorted(BACKUP_ROOT.glob('epoch_*/best.pt')) if BACKUP_ROOT.exists() else []
    assert backups, 'best.pt not found in main or backup paths.'
    BEST = backups[-1]

model = YOLO(str(BEST))
onnx_path = model.export(format='onnx', imgsz=IMGSZ, simplify=True)
print('ONNX:', onnx_path)

easy_best = Path('/kaggle/working/best_micro.pt')
shutil.copy2(str(BEST), str(easy_best))
print('Copied easy download file:', easy_best)

print('Download steps:')
print('1. Click Save Version')
print('2. Open Output tab')
print('3. Download /kaggle/working/best_micro.pt or experiments/yolo/.../best.pt')

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 5, 33600) (49.7 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 202ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 2.57s
Installed 2 packages in 16ms
 + onnxruntime-gpu==1.24.3
 + onnxslim==0.1.87

requirements: AutoUpdate success ✅ 4.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 9.3s, saved as '/kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights/best.onnx' (99.3 MB)

Export complete (14.7s)
Results saved to /kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights
Predict:         yolo predict task=detect model=/kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights/best.onnx imgsz=1280 
Validate:        yolo val task=detect model=/kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights/best.onnx imgsz=1280 data=/kaggle/working/dataset_micro_single_aug/dataset.yaml  
Visualize:       https://netron.app
ONNX: /kaggle/working/experiments/yolo/mp_yolov8m_single_class_micro/weights/best.onnx
Copied easy download file: /kaggle/working/best_micro.pt
Download steps:
1. Click Save Version
2. Open Output tab
3. Download /kaggle/working/best_micro.pt or experiments/yolo/.../best.pt
